## Importing libraries

In [1]:
import numpy as np
import pandas as pd

import json

from pathlib import Path

import pickle

from functions.functions import *

## Importing data

In [2]:
# Dictionary containing all the json files imported (json_file_name : json_content)
json_files_dict = {}

for path in sorted(Path('../raw_data-originals/dati_eumaco_con_pseudo').glob('*.json')):
    try:
        filename = path.stem
        json_content = json.loads(path.read_text()) # Closes automatically the file
        json_files_dict[filename] = json_content
    except json.JSONDecodeError:
        print(f"Skipping invalid JSON: {path}")

# List of all the json files names
json_filenames = list(json_files_dict)

## Make a pretty-print version of all the jsons

In [3]:
# input_dir = '../raw_data-originals/dati_eumaco_con_pseudo'
# output_dir = '../raw_data/jsons-pp-2'

# pretty_print_folder(input_dir, output_dir, indent=2)

## To pandas df

In [4]:
external_fields = [
    'patient_cf' ,
    'assignment_id' ,
    'archivied' ,
    'completed' ,

    'assignment:id' ,
    'assignment:start_at' ,
    'assignment:end_at' ,
    'assignment:active' ,
    'assignment:created_at' ,
    'assignment:updated_at' ,
    'assignment:hub_sn' ,
    'assignment:completed' ,
]

instance_ext_fields = [
    'id' ,
    'type' ,
    'report' ,
    'created_at' ,
    'updated_at' ,
    'registration_time' ,
    'device_sn' ,

# transmission_data
    'transmission_data:device' ,
    'transmission_data:serialnumber' ,
    'transmission_data:timestamp' ,
# battery
    'transmission_data:battery:value' ,
    'transmission_data:battery:unit' ,
# spo2
    'transmission_data:spo2:value' ,
    'transmission_data:spo2:unit' ,
# heartrate
    'transmission_data:heartrate:value' ,
    'transmission_data:heartrate:unit' ,
]

In [5]:
data = json_to_df(json_files_dict, json_filenames,
    just_external = False,
    transm_field = 'SpO2Transmission',
    external_json_fields = external_fields,
    transm_inst_fields = instance_ext_fields
)
df = pd.DataFrame(data)

In [6]:
df

,patient_cf,assignment_id,archivied,completed,assignment:id,assignment:start_at,assignment:end_at,assignment:active,assignment:created_at,assignment:updated_at,...,device_sn,transmission_data:device,transmission_data:serialnumber,transmission_data:timestamp,transmission_data:battery:value,transmission_data:battery:unit,transmission_data:spo2:value,transmission_data:spo2:unit,transmission_data:heartrate:value,transmission_data:heartrate:unit
0,31641425359519401288731,31636,True,False,None,None,None,None,None,None,...,506370467,None,None,None,None,None,None,None,None,None
1,31641425359519401288731,31636,True,False,None,None,None,None,None,None,...,506370467,None,None,None,None,None,None,None,None,None
2,31641425359519401288731,31636,True,False,None,None,None,None,None,None,...,506370467,None,None,None,None,None,None,None,None,None
3,31641425359519401288731,31636,True,False,None,None,None,None,None,None,...,506370467,None,None,None,None,None,None,None,None,None
4,31641425359519401288731,31636,True,False,None,None,None,None,None,None,...,506370467,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8148,41639664657191854858917,33917,True,False,None,None,None,None,None,None,...,506385482,None,None,None,None,None,None,None,None,None
8149,41639664657191854858917,33917,True,False,None,None,None,None,None,None,...,506385482,None,None,None,None,None,None,None,None,None
8150,41639664657191854858917,33917,True,False,None,None,None,None,None,None,...,506385482,None,None,None,None,None,None,None,None,None
8151,41639664657191854858917,33917,True,False,None,None,None,None,None,None,...,506385482,None,None,None,None,None,None,None,None,None


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8153 entries, 0 to 8152
Data columns (total 28 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   patient_cf                         8153 non-null   str   
 1   assignment_id                      8153 non-null   int64 
 2   archivied                          8153 non-null   bool  
 3   completed                          8153 non-null   bool  
 4   assignment:id                      0 non-null      object
 5   assignment:start_at                0 non-null      object
 6   assignment:end_at                  0 non-null      object
 7   assignment:active                  0 non-null      object
 8   assignment:created_at              0 non-null      object
 9   assignment:updated_at              0 non-null      object
 10  assignment:hub_sn                  0 non-null      object
 11  assignment:completed               0 non-null      object
 12  id               

## Analysis and overview

### Fields `assignment_id` and `assignment:id` are always equal to the filename? -> yes

In [9]:
not_equal = []
for json_name in json_filenames:
    assignment_id = json_files_dict[json_name]['assignment_id']
    assignment_id_deep = json_files_dict[json_name]['assignment']['id']
    if int(json_name) == assignment_id == assignment_id_deep:
        continue
    else:
        not_equal.append(json_name)

print(f'Number of jsons where ("json filename" != "assignment_id" != "assignment:id"): {len(not_equal)}')

Number of jsons where ("json filename" != "assignment_id" != "assignment:id"): 0


### Time fields

Dictionary {json_name: {time_field1 : value, ...}, ...}

In [10]:
# Returns a dict where:
# - key is the json file name
# - value is a dict where:
#   * key is the external time field
#   * value is the date value of the field

# sample:
# {json_filename: {
    # 'start_at': date,
    # 'end_at': date,
    # 'created_at': date,
    # 'updated_at': date,
# }}

external_time_values = {}
external_time_fields = ['start_at', 'end_at', 'created_at', 'updated_at']

for json_name in json_filenames:
    external_time_values[json_name] = {}

    for field in external_time_fields:
        value = pd.Timestamp(json_files_dict[json_name]['assignment'][field])
        external_time_values[json_name][field] = value

external_time_values['31636']

{'start_at': Timestamp('2025-01-24 11:19:00+0000', tz='UTC'),
 'end_at': Timestamp('2025-12-31 22:59:00+0000', tz='UTC'),
 'created_at': Timestamp('2025-01-24 11:25:41+0000', tz='UTC'),
 'updated_at': Timestamp('2025-12-18 07:10:22+0000', tz='UTC')}

`start_at` is always lower than the others

In [11]:
# Proves that the field start_at
# always holds the earliest external date value

for json in external_time_values:
    values = external_time_values[json]
    if not(values['start_at'] < min(
        values['end_at'],
        values['created_at'],
        values['updated_at'],
    )):
        print(json)

Dictionary `{json_name : {'min': minimum, 'max': maximum}, ...}` or

Dictionary `{json_name : [minimum, maximum], ...}`

In [12]:
# Returns a dict where:
# - key is the json file name
# - value is a dict where:
#   * key is 'min' or 'max'
#   * value is lowest or highest date value among the external time fields

# sample:
# {json_filename: {
    # 'min': date,
    # 'max': date,
# }}

ext_minmax = {}
for json in external_time_values:
    values = external_time_values[json]
    minimum = values['start_at']
    maximum = values[max(values, key=values.get)]

    # ext_minmax[json] = [minimum, maximum] # Variant for list
    ext_minmax[json] = {'min': minimum, 'max': maximum}

ext_minmax['31636']

{'min': Timestamp('2025-01-24 11:19:00+0000', tz='UTC'),
 'max': Timestamp('2025-12-31 22:59:00+0000', tz='UTC')}

In [13]:
def internal_btw_external(json_files_dict, json_filenames, transm_to_check, ext_minmax, int_mid_timefield, *, ids=False):
    '''
    Takes in input:
    - a dictionary {json_filename: json_file_content}
    - a list of json filenames
    - the transmission subfield to check
    - a dictionary with all the minimum and maximum dates for every json file
    - an "internal" timefield to check

    Check if the int_mid_timefield stays between min and max dates for every json file

    Returns an overview dictionary {'problems': {json_file: [position, ...]}, 'missing': [json_files]} that tells:
    - given a json file which have the field different from the sample one:
        - the instance's position within the file, if instance == False (default behaviour)
        - the instance's if field, if ids == True
        - the actual instance structure, if instance == True
    - which json files misses the transmission subfield to check
    '''

    # midtime = ['created_at', 'updated_at', 'registration_time', 'timestamp']

    # The returned object will have this structure:
    # {'different': {json_file: [position, ...]}, 'missing': [json_files]}
    overview = {'problems': {}, 'missing': []}

    # Iterates over the jsons
    for json_name in json_filenames:

        # transmissions will be a dict that contains the actual "transmissions" field of the json
        # {transm_to_check: content}
        transmissions = json_files_dict[json_name].get('transmissions', {})

        # If the transmission field to check is NOT present:
        # Append the json name to the missing list ...
        # .. and go directly to the next json file
        if transm_to_check not in transmissions:
            overview['missing'].append(json_name)
            continue

        # If the transmission field to check is present in the json:

        # Place in instances the complete list of instances of the transm_to_check
        instances = transmissions[transm_to_check][0]

        # Iterate over the long list of the instances and set:
        # - t_position as the transmission field instance's position
        # - t_instance as the the transmission field's instance itself
        for t_position, t_instance in enumerate(instances):

            if int_mid_timefield == 'timestamp':
                midtime = t_instance['transmission_data'][int_mid_timefield]
            else:
                midtime = t_instance[int_mid_timefield]


            condition = (
                ext_minmax[json_name]['min'] <
                # pd.Timestamp(t_instance['transmission_data']['timestamp']) <
                pd.Timestamp(midtime) <
                ext_minmax[json_name]['max']
            )
            if condition == False:
                if ids == False:
                    overview['problems'].setdefault(json_name, []).append(t_position)
                else:
                    overview['problems'].setdefault(json_name, []).append(t_instance['id'])

    return overview

In [14]:
field_to_check = 'SpO2Transmission'

In [15]:
internals = internal_btw_external(json_files_dict, json_filenames, field_to_check, ext_minmax, 'timestamp')
internals

{'problems': {'31696': [0],
  '31893': [290,
   291,
   292,
   293,
   294,
   295,
   296,
   297,
   298,
   299,
   300,
   301,
   302,
   303,
   304,
   305,
   306,
   307,
   308,
   309,
   310,
   311],
  '32323': [202,
   203,
   204,
   205,
   206,
   207,
   208,
   209,
   210,
   211,
   212,
   213,
   214,
   215,
   216,
   217,
   218,
   219,
   220,
   221,
   222,
   223,
   224,
   225,
   226,
   227,
   228,
   229,
   230,
   231,
   232,
   233,
   234,
   235,
   236,
   237],
  '32774': [0, 1, 2, 3, 4],
  '33118': [0],
  '33467': [0]},
 'missing': ['31964']}

### Check if the "external" fields except the `transmissions` have the consistent values

#### booleans

##### assignment_ids

In [16]:
column1 = 'assignment_id'
column2 = 'assignment:id'
print(columns_consistency(df, column1, column2, return_number=True))

KeyError: 0

##### archived/completed

In [ ]:
column1 = 'archivied'
column2 = 'completed'

print(columns_consistency(df, column2, column1, return_number=True))

for json in columns_consistency(df, column2, column1).index:
    print(f'{json_filenames[json]}.json')

##### completed/assignment:completed

In [ ]:
column1 = 'completed'
column2 = 'assignment:completed'

print(columns_consistency(df, column2, column1, return_number=True))

##### assignment:active/assignment:completed

In [ ]:
column1 = 'assignment:active'
column2 = 'assignment:completed'

print(columns_consistency(df, column2, column1, return_number=True))

for json in columns_consistency(df, column2, column1).index:
    print(f'{json_filenames[json]}.json')

##### archivied/assignment:completed

In [ ]:
column1 = 'archivied'
column2 = 'assignment:completed'

print(columns_consistency(df, column2, column1, return_number=True))

for json in columns_consistency(df, column2, column1).index:
    print(f'{json_filenames[json]}.json')

## Parsing and reformatting

## Export

In [ ]:
# Pickle
with open('../raw_data/pickles/4-eumaco-json_files_dict.pickle', 'wb') as file:
    pickle.dump(json_files_dict, file, protocol=pickle.HIGHEST_PROTOCOL)

# # Parquet
# df.to_parquet('../raw_data/parquets/4-eumaco-json_pseudopaz.parquet')